# ADRec Baby: authors' pretrained embeddings and warm-up

Train ten fresh ADRec models: seeds 42–46 with five warm-up epochs and with zero warm-up epochs. Every run uses the same original Baby `pretrain.pth` from the authors' pinned repository commit. Only its item embeddings are loaded; the ADRec backbone is freshly initialized for each run. No pretraining is repeated.

The result comparison has four conditions: project embeddings with 5/0 warm-up epochs (historical results), and authors' embeddings with 5/0 epochs (new runs). All use 32 diffusion steps. Full training configurations match the corresponding project baseline; the embedding source is the experimental change.

1. Upload `adrec_baby_author_embeddings_inputs.zip` as a private Kaggle dataset and attach it here. It includes the code, Baby dataset, authors' checkpoint, and historical comparison records.
2. Enable Internet and a GPU.
3. Run all cells. `RUN_FULL = True` runs both smoke checks followed by all ten full runs; `False` runs smoke checks only.

The authors' checkpoint has no recorded pretraining seed. The five new seeds vary downstream ADRec training while fixing pretrained embeddings; the historical project runs also varied pretraining. This experiment evaluates this released checkpoint under our trainer and is not a guaranteed reproduction of the paper.


In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
INPUT_BUNDLE = None  # Optional exact path to this experiment's ZIP or extracted folder.
RESUME_ROOT = None  # Folder containing authors_warmup5_seed42/, authors_warmup0_seed42/, etc.
RUN_FULL = True
SEEDS = [42, 43, 44, 45, 46]
WARMUPS = [5, 0]
WORK_ROOT.mkdir(parents=True, exist_ok=True)


## 1. Verify and load inputs

The setup supports intact ZIPs and Kaggle-extracted folders. Hashes verify every input and the code snapshot. Use `INPUT_BUNDLE` if multiple copies are attached. Start a fresh kernel when changing the source bundle.


In [ ]:
import hashlib
import json
import shutil
import sys
import zipfile


def read_json(path):
    return json.loads(Path(path).read_text())


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2) + '\n')
    temporary.replace(path)


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def inside(root, name):
    root = Path(root).resolve()
    path = (root / name).resolve()
    if path == root or root not in path.parents:
        raise ValueError(f'Unsafe archive or manifest path: {name}')
    return path


def unpack(archive, destination):
    destination = Path(destination)
    with zipfile.ZipFile(archive) as bundle:
        for item in bundle.infolist():
            inside(destination, item.filename)
            if (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Archive links are not supported')
        bundle.extractall(destination)
    return destination


def locate_inputs(root, override, cache):
    root = Path(override) if override else Path(root)
    if root.is_file():
        root = unpack(root, cache)
    manifests = sorted(root.rglob('author_inputs_manifest.json'))
    if not manifests and override is None:
        archives = sorted(root.rglob('adrec_baby_author_embeddings_inputs.zip'))
        if len(archives) != 1:
            raise ValueError('Attach one author-embedding input bundle or set INPUT_BUNDLE.')
        root = unpack(archives[0], cache)
        manifests = sorted(root.rglob('author_inputs_manifest.json'))
    if len(manifests) != 1:
        raise ValueError(f'Expected one author-embedding manifest; found {manifests}')
    manifest = read_json(manifests[0])
    if manifest['experiment'] != 'adrec_baby_author_embeddings' or manifest['seeds'] != [42, 43, 44, 45, 46]:
        raise ValueError('Wrong experiment bundle')
    root = manifests[0].parent
    for name, expected in manifest['files'].items():
        if sha256(inside(root, name)) != expected:
            raise ValueError(f'Input hash mismatch: {name}')
    return root, manifest


def prepare_code(inputs, repo, manifest):
    inputs, repo = Path(inputs), Path(repo)
    if repo.exists():
        raise FileExistsError('Source directory already exists. Restart with a fresh WORK_ROOT or reuse the verified REPO variable.')
    manifests = sorted(inputs.rglob('bundle_manifest.json'))
    if len(manifests) == 1:
        shutil.copytree(manifests[0].parent, repo)
    elif not manifests and (inputs / 'cmpe257_kaggle_code.zip').is_file():
        unpack(inputs / 'cmpe257_kaggle_code.zip', repo)
    else:
        raise ValueError('Expected one source ZIP or one extracted source manifest')
    if sha256(repo / 'bundle_manifest.json') != manifest['code_manifest_sha256']:
        raise ValueError('Wrong code bundle')
    source = read_json(repo / 'bundle_manifest.json')
    for name, expected in source['files'].items():
        if sha256(inside(repo, name)) != expected:
            raise ValueError(f'Source hash mismatch: {name}')
    return repo


In [ ]:
INPUTS, INPUT_MANIFEST = locate_inputs(INPUT_ROOT, INPUT_BUNDLE, WORK_ROOT / 'author_input_cache')
REPO_PATH = WORK_ROOT / 'CMPE257_ML_author_embeddings'
if not REPO_PATH.exists():
    REPO = prepare_code(INPUTS, REPO_PATH, INPUT_MANIFEST)
else:
    REPO = REPO_PATH
    if sha256(REPO / 'bundle_manifest.json') != INPUT_MANIFEST['code_manifest_sha256']:
        raise ValueError('Existing code is from a different bundle; start a fresh session.')
    for name, expected in read_json(REPO / 'bundle_manifest.json')['files'].items():
        if sha256(inside(REPO, name)) != expected:
            raise ValueError(f'Existing source changed: {name}')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
DATASET_PATH = INPUTS / 'dataset.pkl'
PROJECT_FIVE = INPUTS / 'baseline/project_warmup5'
PROJECT_ZERO = INPUTS / 'baseline/project_warmup0'
AUTHOR_CHECKPOINT = INPUTS / 'authors/baby/pretrain.pth'
print('Code:', REPO)
print('Data:', DATASET_PATH)
print('ADRec seeds:', INPUT_MANIFEST['seeds'])
print('Authors checkpoint:', AUTHOR_CHECKPOINT)


## 2. Install dependencies and check the runtime

Install the shared requirements, retaining a compatible CUDA-enabled PyTorch installation. Restart the kernel if requested, then run from the beginning. Full training uses one GPU.


In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements-shared.txt')])


In [ ]:
import torch
from src.training.kaggle import train, make_config
from src.data import load_dataset
from src.data.load import dataset_summary

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
if RUN_FULL and not torch.cuda.is_available():
    raise RuntimeError('Select a GPU for full runs, or set RUN_FULL = False for a CPU smoke check.')
print(dataset_summary(load_dataset(DATASET_PATH, INPUT_MANIFEST['dataset_sha256'])))


## 3. Verify the checkpoint and historical comparisons

The authors' file must match its pinned Git blob and SHA-256, have 4,732 embedding rows (including padding) and 128 columns, and contain finite values. Dataset identity is established by the official Baby checkpoint path and released Baby dataset at the same pinned commit; the checkpoint has no separate item-mapping or training metadata.

The original five-epoch and zero-epoch results must use matching per-seed project embeddings and compatible source/configuration settings. Only the previously audited Baby source migration is accepted. These checks preserve our existing training and evaluation behavior, including strict embedding freezing during warm-up.

Resume checks require identical settings, authors' checkpoint, and source bundle. Completed runs are reused; unfinished runs resume from `latest.pt`.


In [ ]:
import csv
import math
import statistics
from src.training.kaggle import build_model
from src.utils.reproducibility import seed_everything

METRICS = ['HR@5', 'HR@10', 'HR@20', 'NDCG@5', 'NDCG@10', 'NDCG@20']
METADATA = ['config.json', 'results.json', 'history.json', 'environment.json', 'requirements-lock.txt', 'experiment_input.json']
RUNS_ROOT = WORK_ROOT / 'runs/adrec_author_embeddings/baby'
RESULTS_DIR = WORK_ROOT / 'results/adrec_author_embeddings/baby'
ARCHIVE_ROOT = WORK_ROOT / 'exports'
ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)


def variant_config(base, warmup, smoke=False):
    if warmup not in (5, 0):
        raise ValueError('Warm-up must be 5 or 0')
    cfg = dict(base, warmup_epochs=warmup)
    if smoke:
        # Preserve the actual five-epoch warm-up and pretrained embedding shape.
        cfg.update(mode='smoke', user_limit=9, train_batch_size=4,
                   evaluation_batch_size=4, epochs=6)
    return cfg


def check_report(cfg, report):
    test = report.get('test')
    if report['model'] != 'adrec' or report['mode'] != cfg['mode']:
        raise ValueError('Result model or mode mismatch')
    if not test or test['protocol'] != 'adrec_upstream' or test['units'] != 'percent':
        raise ValueError('Expected completed upstream-protocol test evaluation')
    users = cfg['user_limit'] or 11761
    if report['users'] != users or test['users'] != users:
        raise ValueError('Wrong evaluation population')
    for metric in METRICS:
        if not math.isfinite(test['metrics'][metric]) or not 0 <= test['metrics'][metric] <= 100:
            raise ValueError(f'Invalid metric: {metric}')


def check_source(env):
    current = read_json(REPO / 'bundle_manifest.json')['files']
    allowed = read_json(REPO / 'configs/legacy_source_compatibility.json')['migrations']
    migrations = []
    for name, old in env['source']['files'].items():
        relevant = ((name.startswith(('src/', 'third_party/ADRec/src/')) and name.endswith(('.py', '.yaml')))
                    or name in ('configs/shared.json', 'configs/kaggle.json'))
        if not relevant:
            continue
        new = current.get(name)
        if old != new:
            matches = [m for m in allowed if m['dataset'] == 'baby' and m['path'] == name
                       and m['before'] == old and m['after'] == new]
            if len(matches) != 1:
                raise ValueError(f'Unreviewed source difference: {name}')
            migrations.append(matches[0])
    return migrations


def validate_author_checkpoint(path, spec):
    raw = Path(path).read_bytes()
    if (len(raw) != spec['size_bytes'] or hashlib.sha256(raw).hexdigest() != spec['sha256']
            or hashlib.sha1(f'blob {len(raw)}\0'.encode() + raw).hexdigest() != spec['git_blob_sha1']):
        raise ValueError('Authors checkpoint does not match the pinned official file')
    saved = torch.load(path, map_location='cpu', weights_only=True)
    weight = saved[spec['state_key']]
    if list(weight.shape) != spec['embedding_shape'] or str(weight.dtype) != spec['dtype'] or not torch.isfinite(weight).all():
        raise ValueError('Invalid authors embedding shape, dtype, or values')
    return weight


def validate_references():
    if SEEDS != [42, 43, 44, 45, 46] or WARMUPS != [5, 0]:
        raise ValueError('This experiment requires seeds 42–46 and both warm-up settings')
    spec = read_json(REPO / 'configs/author_baby_embeddings.json')
    if read_json(INPUTS / 'author_checkpoint.json') != spec:
        raise ValueError('Author checkpoint provenance mismatch')
    if spec['dataset_sha256'] != sha256(DATASET_PATH):
        raise ValueError('Wrong Baby dataset')
    validate_author_checkpoint(AUTHOR_CHECKPOINT, spec)
    if not read_json(PROJECT_ZERO / 'completion.json')['complete']:
        raise ValueError('Historical no-warm-up comparison is incomplete')
    refs, migrations = {}, []
    for seed in SEEDS:
        base = read_json(PROJECT_FIVE / f'adrec_seed{seed}_config.json')
        expected = make_config('adrec', 'full')
        expected['seed'] = seed
        if base != expected or base['warmup_epochs'] != 5 or base['diffusion_steps'] != 32:
            raise ValueError(f'Original configuration mismatch: {seed}')
        reports = {}
        for warmup, folder in [(5, PROJECT_FIVE), (0, PROJECT_ZERO)]:
            cfg = read_json(folder / f'adrec_seed{seed}_config.json')
            if cfg != variant_config(base, warmup):
                raise ValueError(f'Historical configuration mismatch: seed {seed}, warm-up {warmup}')
            report = read_json(folder / f'adrec_seed{seed}_results.json')
            check_report(cfg, report)
            env = read_json(folder / f'adrec_seed{seed}_environment.json')
            migrations.extend({'seed': seed, 'historical_warmup': warmup, **m} for m in check_source(env))
            reports[warmup] = report
        original_hash = read_json(PROJECT_FIVE / f'adrec_seed{seed}_checkpoints.json')['files']['pretrain/best.pt']
        zero_hash = read_json(PROJECT_ZERO / f'adrec_seed{seed}_checkpoints.json')['pretrained_sha256']
        if original_hash != zero_hash:
            raise ValueError('Historical conditions used different pretrained embeddings')
        refs[seed] = {'config': base, 'reports': reports, 'pretrained_sha256': original_hash}
    return refs, migrations, spec


def ensure_run(cfg, output, prior=None):
    output = Path(output)
    identity = {'config': cfg, 'embedding_source': 'authors',
                'pretrained_sha256': sha256(AUTHOR_CHECKPOINT),
                'code_manifest_sha256': INPUT_MANIFEST['code_manifest_sha256']}
    if not (output / 'latest.pt').exists() and prior and Path(prior).is_dir():
        if output.exists() and any(output.iterdir()):
            raise ValueError(f'Resume destination is not empty: {output}')
        shutil.copytree(prior, output, dirs_exist_ok=True)
    output.mkdir(parents=True, exist_ok=True)
    marker = output / 'experiment_input.json'
    if marker.exists() and read_json(marker) != identity:
        raise ValueError(f'Run input mismatch: {output}')
    resume = output / 'latest.pt'
    if resume.exists() and not marker.exists():
        raise ValueError('Saved run is missing input provenance')
    write_json(marker, identity)
    if resume.exists():
        if read_json(output / 'config.json') != cfg:
            raise ValueError('Resume configuration mismatch')
        result_path = output / 'results.json'
        result = read_json(result_path) if result_path.exists() else None
        if result and (result['early_stopped'] or result['epochs_completed'] >= cfg['epochs']) and result.get('test') and (output / 'best.pt').is_file():
            check_report(cfg, result)
            print('Reusing completed run:', output.name)
            return result
    result = train(cfg, DATASET_PATH, output, pretrained_checkpoint=AUTHOR_CHECKPOINT,
                   resume=resume if resume.exists() else None, device=DEVICE, finalize=True)
    check_report(cfg, result)
    return result


def write_csv(path, columns, rows):
    with Path(path).open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


def result_row(source, warmup, seed, result):
    return {'embedding_source': source, 'warmup_epochs': warmup, 'seed': seed,
            'model': 'ADRec', 'dataset': 'Baby', 'best_epoch': result['best_epoch'],
            'epochs_completed': result['epochs_completed'], **result['test']['metrics']}


def summarize(rows):
    output = []
    for source, warmup in sorted({(r['embedding_source'], r['warmup_epochs']) for r in rows}):
        group = sorted([r for r in rows if r['embedding_source'] == source and r['warmup_epochs'] == warmup], key=lambda r: r['seed'])
        common = {'embedding_source': source, 'warmup_epochs': warmup, 'model': 'ADRec', 'dataset': 'Baby',
                  'n_runs': len(group), 'seeds': ','.join(str(r['seed']) for r in group)}
        output.append({**common, 'statistic': 'mean',
                       **{m: round(statistics.mean(r[m] for r in group), 4) for m in METRICS}})
        if len(group) > 1:
            output.append({**common, 'statistic': 'std (sample, n-1)',
                           **{m: round(statistics.stdev(r[m] for r in group), 4) for m in METRICS}})
    return output


def export_results():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    completed = {}
    for seed in SEEDS:
        for warmup in WARMUPS:
            run = RUNS_ROOT / f'authors_warmup{warmup}_seed{seed}'
            if not (run / 'results.json').is_file():
                continue
            cfg = read_json(run / 'config.json')
            expected = variant_config(REFERENCES[seed]['config'], warmup)
            if cfg != expected:
                raise ValueError('Export requires matching full-run configuration')
            report = read_json(run / 'results.json')
            check_report(cfg, report)
            if not (report['early_stopped'] or report['epochs_completed'] >= cfg['epochs']):
                continue
            identity = read_json(run / 'experiment_input.json')
            if identity['pretrained_sha256'] != AUTHOR_SPEC['sha256'] or identity['code_manifest_sha256'] != INPUT_MANIFEST['code_manifest_sha256']:
                raise ValueError('Run provenance does not match this experiment')
            target = RESULTS_DIR / f'authors_warmup{warmup}'
            target.mkdir(exist_ok=True)
            for name in METADATA:
                shutil.copyfile(run / name, target / f'adrec_seed{seed}_{name}')
            write_json(target / f'adrec_seed{seed}_checkpoints.json', {
                'seed': seed, 'warmup_epochs': warmup, 'pretrained_sha256': AUTHOR_SPEC['sha256'],
                'location': f'checkpoints/adrec_author_embeddings/baby/{run.name}',
                'files': {name: sha256(run / name) for name in ['best.pt', 'latest.pt']}})
            completed[(seed, warmup)] = report
    # Save every completed run, but compare four conditions only on paired seeds.
    cols = ['embedding_source','warmup_epochs','seed','model','dataset','best_epoch','epochs_completed',*METRICS]
    new_rows = [result_row('authors', w, s, report) for (s,w),report in sorted(completed.items())]
    write_csv(RESULTS_DIR / 'completed_author_runs.csv', cols, new_rows)
    paired_seeds = [s for s in SEEDS if all((s,w) in completed for w in WARMUPS)]
    comparison, differences = [], []
    for seed in paired_seeds:
        for warmup in WARMUPS:
            comparison.extend([result_row('authors', warmup, seed, completed[(seed,warmup)]),
                               result_row('project', warmup, seed, REFERENCES[seed]['reports'][warmup])])
            differences.append({'comparison': f'authors_minus_project_warmup{warmup}', 'seed': seed,
                **{m: round(completed[(seed,warmup)]['test']['metrics'][m] - REFERENCES[seed]['reports'][warmup]['test']['metrics'][m],4) for m in METRICS}})
        for source in ['authors','project']:
            pair = {w: completed[(seed,w)] if source == 'authors' else REFERENCES[seed]['reports'][w] for w in WARMUPS}
            differences.append({'comparison': f'{source}_warmup0_minus_warmup5', 'seed': seed,
                **{m: round(pair[0]['test']['metrics'][m] - pair[5]['test']['metrics'][m],4) for m in METRICS}})
    write_csv(RESULTS_DIR / 'accuracy_runs.csv', cols, comparison)
    write_csv(RESULTS_DIR / 'accuracy_summary.csv', ['embedding_source','warmup_epochs','model','dataset','n_runs','seeds','statistic',*METRICS], summarize(comparison))
    write_csv(RESULTS_DIR / 'paired_differences.csv', ['comparison','seed',*METRICS], differences)
    write_json(RESULTS_DIR / 'completion.json', {'expected_seeds': SEEDS, 'expected_warmups': WARMUPS,
               'completed_runs': [{'seed':s,'warmup_epochs':w} for s,w in sorted(completed)],
               'paired_seeds':paired_seeds, 'complete':len(completed)==len(SEEDS)*len(WARMUPS)})
    shutil.copytree(INPUTS / 'baseline', RESULTS_DIR / 'baseline', dirs_exist_ok=True)
    write_json(RESULTS_DIR / 'author_checkpoint.json', AUTHOR_SPEC)
    write_json(RESULTS_DIR / 'source_migrations.json', SOURCE_MIGRATIONS)
    write_json(RESULTS_DIR / 'input_manifest.json', INPUT_MANIFEST)
    write_json(RESULTS_DIR / 'smoke_check.json', SMOKE_CHECK)
    write_csv(RESULTS_DIR / 'paper_reference.csv', ['warmup_epochs','HR@20','NDCG@20','source'], [
        {'warmup_epochs':5,'HR@20':7.1524,'NDCG@20':3.1455,'source':'https://arxiv.org/html/2505.19544v1#S4.SS4'},
        {'warmup_epochs':0,'HR@20':6.7507,'NDCG@20':2.9535,'source':'https://arxiv.org/html/2505.19544v1#S4.SS4'}])
    (RESULTS_DIR / 'EXPERIMENT.md').write_text(EXPERIMENT_TEXT)
    return comparison


def save_archives():
    paths = []
    if RESULTS_DIR.exists():
        paths.append(shutil.make_archive(str(ARCHIVE_ROOT / 'adrec_baby_author_embeddings_results'), 'zip', root_dir=RESULTS_DIR))
    if RUNS_ROOT.exists():
        paths.append(shutil.make_archive(str(ARCHIVE_ROOT / 'adrec_baby_author_embeddings_models'), 'zip', root_dir=RUNS_ROOT))
    return paths

EXPERIMENT_TEXT = "# Baby ADRec: authors' embeddings and warm-up\n\nNew runs use the pinned original authors' Baby embedding checkpoint for both five-epoch and zero-epoch warm-up, with downstream seeds 42–46. Only item_embedding.weight initializes ADRec; the backbone starts fresh. No embedding pretraining is repeated. All full configurations match their corresponding historical project conditions, including 32 diffusion steps and validation-based checkpoint selection.\n\nFour conditions are compared: authors/project embeddings, each with warmup_epochs=5/0. Historical project scores are reused, not re-evaluated or retrained. accuracy_runs.csv and accuracy_summary.csv use only seeds with both new conditions complete, ensuring identical seed coverage for all four groups. completed_author_runs.csv includes every completed new run. completion.json records coverage. paired_differences.csv reports authors minus project for each warm-up setting, and warmup0 minus warmup5 within each embedding source, in percentage points. Mean and sample standard deviation are computed separately by condition; a single completed seed has no standard deviation.\n\nAll new runs share one fixed released authors' checkpoint, whose pretraining seed is unrecorded. Historical project runs used independently pretrained per-seed embeddings. Their standard deviations therefore include different sources of randomness. This comparison measures the effect of substituting this particular released checkpoint under our trainer, not all possible differences in pretraining procedures. Runtime changes and the documented trainer differences remain relevant. Our warm-up strictly freezes embeddings, unlike possible weight-decay updates through the released optimizer. Source compatibility checks and accepted migrations are recorded separately. This is not an exact replication of the paper's training pipeline, and the release does not establish that this file is the checkpoint used for every reported paper run.\n\npaper_reference.csv contains published Baby point estimates from Table 4, https://arxiv.org/html/2505.19544v1#S4.SS4. They are references, not measurements from this notebook; no paper per-seed scores or standard deviations are invented. No inference-speed benchmark is performed. Weights are stored separately from result metadata. Checkpoint provenance is in author_checkpoint.json.\n"


In [ ]:
REFERENCES, SOURCE_MIGRATIONS, AUTHOR_SPEC = validate_references()
print('Verified authors checkpoint SHA-256:', AUTHOR_SPEC['sha256'])
print('Verified both historical conditions for seeds:', SEEDS)
for warmup in WARMUPS:
    print('New condition: authors embeddings, warmup_epochs =', warmup,
          ', diffusion_steps =', variant_config(REFERENCES[42]['config'], warmup)['diffusion_steps'])


## 4. Smoke checks for both warm-up settings

Each smoke run uses nine users for six epochs with the original 128-dimensional model and 32 diffusion steps. The five-epoch condition freezes embeddings for epochs 0–4 and unfreezes them at epoch 5; the zero-epoch condition trains them immediately. The checks verify exact loading of the authors' embeddings, identical initial backbone weights between conditions at the same seed, expected training flags, and changed embeddings after training. Smoke scores never enter the full comparison.


In [ ]:
smoke_checks = []
initial_embeddings = validate_author_checkpoint(AUTHOR_CHECKPOINT, AUTHOR_SPEC)
initial_state = None
for warmup in WARMUPS:
    cfg = variant_config(REFERENCES[42]['config'], warmup, smoke=True)
    seed_everything(42)
    initial_model = build_model(cfg, 4731, torch.device('cpu'), AUTHOR_CHECKPOINT)
    if not torch.equal(initial_model.item_embedding.weight.detach(), initial_embeddings):
        raise AssertionError('Authors embeddings were not loaded exactly')
    if initial_state is None:
        initial_state = {k:v.detach().clone() for k,v in initial_model.state_dict().items()}
    elif not all(torch.equal(v, initial_state[k]) for k,v in initial_model.state_dict().items()):
        raise AssertionError('Initial model states differ between warm-up conditions')
    del initial_model
    output = WORK_ROOT / f'smoke_runs/adrec_author_embeddings/baby/authors_warmup{warmup}_seed42'
    ensure_run(cfg, output)
    latest = torch.load(output / 'latest.pt', map_location='cpu', weights_only=True)
    history = latest['history']
    if len(history) != 6 or [h['embedding_trainable'] for h in history] != [e >= warmup for e in range(6)]:
        raise AssertionError('Unexpected embedding training schedule')
    change = torch.linalg.vector_norm(latest['model_state']['item_embedding.weight'] - initial_embeddings).item()
    if not math.isfinite(change) or change <= 0:
        raise AssertionError('Embeddings did not update after becoming trainable')
    smoke_checks.append({'warmup_epochs':warmup, 'seed':42, 'passed':True,
                         'embedding_change_l2':change, 'config':cfg})
    del latest
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
del initial_state, initial_embeddings
SMOKE_CHECK = {'passed':True, 'checks':smoke_checks, 'pretrained_sha256':AUTHOR_SPEC['sha256'],
               'code_manifest_sha256':INPUT_MANIFEST['code_manifest_sha256']}
write_json(WORK_ROOT / 'smoke_runs/adrec_author_embeddings/baby/smoke_check.json', SMOKE_CHECK)
print('Both smoke checks passed; full-run configurations remain unchanged.')


## 5. Train ten runs and export results

For each seed, train the five-epoch condition, then the zero-epoch condition. Each call resets the same downstream seed and loads the same authors' embeddings. The optimizer, losses, learning rate, diffusion steps, data splits, validation schedule, and evaluation protocol remain fixed.

Archives refresh after every completed run. Rerunning this cell skips completed runs and resumes interrupted runs. For a new session, attach the extracted model archive and set `RESUME_ROOT` to the folder directly containing `authors_warmup5_seed42/`, `authors_warmup0_seed42/`, and the other runs. Use the same input bundle.


In [ ]:
if RUN_FULL:
    if not SMOKE_CHECK['passed']:
        raise RuntimeError('Run both smoke checks first.')
    for seed in SEEDS:
        for warmup in WARMUPS:
            output = RUNS_ROOT / f'authors_warmup{warmup}_seed{seed}'
            prior = Path(RESUME_ROOT) / output.name if RESUME_ROOT else None
            try:
                result = ensure_run(variant_config(REFERENCES[seed]['config'], warmup), output, prior)
                export_results()
                print('Completed:', output.name, result['test']['metrics'])
            finally:
                for path in save_archives():
                    print('Saved:', path)
else:
    print('Smoke-only execution finished. Set RUN_FULL = True to run all ten models.')


## 6. Inspect and download outputs

`accuracy_summary.csv` compares four conditions using the same completed paired seeds. `completed_author_runs.csv` also lists individual new runs that do not yet have their matching condition. `completion.json` must list all ten runs and `complete: true` for the final comparison. The paper reference is separate from measured results.

Download both ZIPs from `/kaggle/working/exports/`:

- `adrec_baby_author_embeddings_results.zip` → extract into `results/adrec_author_embeddings/baby/` locally. Contains configurations, provenance, histories, metrics, CSV summaries, and historical reference metadata; no weights.
- `adrec_baby_author_embeddings_models.zip` → extract into `checkpoints/adrec_author_embeddings/baby/` locally. Contains `best.pt` for inference and `latest.pt` for resuming each run.

Keep weights outside Git. The notebook creates archives but does not download them to your laptop. Kaggle **Save Version → Save & Run All** starts a fresh background session; it does not inherit interactive output files. Attach saved model folders through `RESUME_ROOT` to resume existing work.


In [ ]:
try:
    from IPython.display import FileLink, display
except ImportError:
    FileLink = str
    display = print

summary_path = RESULTS_DIR / 'accuracy_summary.csv'
if summary_path.exists():
    print(summary_path.read_text())
    print('Completion:', read_json(RESULTS_DIR / 'completion.json'))
    print('Paper reference (percent):')
    print((RESULTS_DIR / 'paper_reference.csv').read_text())
for path in save_archives():
    display(FileLink(path))
